# 🎙️ Modern Whisper Transcription Pipeline

Pipeline moderno de transcripción de audio usando:
- **`faster-whisper`** — Motor principal (2-4x más rápido que el Whisper original)
- **`WhisperX`** — Timestamps por palabra y diarización de hablantes
- **Exportación** a `.txt`, `.json`, `.srt` y `.vtt`

---
### ⚡ Requisitos
Asegúrate de tener activado el runtime de GPU: `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU`

## 1️⃣ Instalación de dependencias

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Core dependencies
# ─────────────────────────────────────────────────────────────────────────────
!pip install -q faster-whisper
!pip install -q whisperx
!pip install -q srt
!sudo apt-get install -q -y ffmpeg

print("✅ All dependencies installed.")

## 2️⃣ Configuración global

Ajusta los parámetros según tu caso de uso antes de continuar.

In [ ]:
import torch

# ─────────────────────────────────────────────────────────────────────────────
# Global configuration
# ─────────────────────────────────────────────────────────────────────────────

# Model options: "tiny", "base", "small", "medium", "large-v2", "large-v3", "large-v3-turbo"
# "large-v3-turbo" offers the best speed/accuracy tradeoff for production use
MODEL_SIZE = "large-v3-turbo"

# Language code (e.g., "es" for Spanish, "en" for English, None for auto-detect)
LANGUAGE = None  # Set to None to enable automatic language detection

# Device and precision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

# Beam size — higher = more accurate but slower (5 is a good default)
BEAM_SIZE = 5

# Enable speaker diarization (requires HuggingFace token — see Section 5)
ENABLE_DIARIZATION = False
HUGGINGFACE_TOKEN = ""  # Required only when ENABLE_DIARIZATION = True

print(f"🖥️  Device: {DEVICE} ({COMPUTE_TYPE})")
print(f"🤖  Model: {MODEL_SIZE}")
print(f"🌐  Language: {'Auto-detect' if LANGUAGE is None else LANGUAGE}")
print(f"👥  Diarization: {'Enabled' if ENABLE_DIARIZATION else 'Disabled'}")

## 3️⃣ Subir archivo de audio

Formatos soportados: `.mp3`, `.mp4`, `.wav`, `.m4a`, `.ogg`, `.flac`, `.webm`

In [ ]:
from google.colab import files
import os

# Upload audio file
print("📁 Select your audio file:")
uploaded = files.upload()

AUDIO_FILE = list(uploaded.keys())[0]
file_size_mb = os.path.getsize(AUDIO_FILE) / (1024 * 1024)

print(f"\n✅ File loaded: {AUDIO_FILE}")
print(f"📦  Size: {file_size_mb:.2f} MB")

## 4️⃣ Transcripción con faster-whisper

`faster-whisper` usa CTranslate2 como backend, lo que lo hace 2-4x más rápido que el
Whisper original de OpenAI manteniendo la misma precisión.

In [ ]:
from faster_whisper import WhisperModel
import time

# ─────────────────────────────────────────────────────────────────────────────
# Load model
# ─────────────────────────────────────────────────────────────────────────────
print(f"⏳ Loading model '{MODEL_SIZE}'...")
model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)
print("✅ Model loaded.\n")

# ─────────────────────────────────────────────────────────────────────────────
# Transcribe
# ─────────────────────────────────────────────────────────────────────────────
print("🎙️  Transcribing...")
start_time = time.time()

segments_generator, transcription_info = model.transcribe(
    AUDIO_FILE,
    beam_size=BEAM_SIZE,
    language=LANGUAGE,
    word_timestamps=True,  # Enable word-level timestamps
    vad_filter=True,       # Voice Activity Detection — skips silence automatically
    vad_parameters=dict(
        min_silence_duration_ms=500
    )
)

# Collect all segments (generator is lazy, must be iterated)
segments = list(segments_generator)

elapsed = time.time() - start_time

# ─────────────────────────────────────────────────────────────────────────────
# Build full transcription text
# ─────────────────────────────────────────────────────────────────────────────
full_text = " ".join(seg.text.strip() for seg in segments)

# ─────────────────────────────────────────────────────────────────────────────
# Print summary
# ─────────────────────────────────────────────────────────────────────────────
audio_duration = transcription_info.duration
speed_factor = audio_duration / elapsed if elapsed > 0 else 0

print(f"\n📊  Transcription complete!")
print(f"🌐  Language detected: {transcription_info.language} "
      f"(confidence: {transcription_info.language_probability:.1%})")
print(f"⏱️   Audio duration: {audio_duration:.1f}s")
print(f"⚡  Processing time: {elapsed:.1f}s  ({speed_factor:.1f}x real-time)")
print(f"📝  Segments found: {len(segments)}")
print("\n" + "─" * 60)
print("FULL TRANSCRIPTION:")
print("─" * 60)
print(full_text)

## 5️⃣ Diarización de hablantes con WhisperX *(opcional)*

Esta sección identifica **quién habló y cuándo**. Requiere:
1. Una cuenta en [huggingface.co](https://huggingface.co)
2. Aceptar los términos de uso del modelo `pyannote/speaker-diarization-3.1`
3. Ingresar tu token HuggingFace en `HUGGINGFACE_TOKEN` en la sección de configuración

Si no necesitas diarización, salta directamente a la sección de exportación.

In [ ]:
if not ENABLE_DIARIZATION:
    print("ℹ️  Diarization is disabled. Set ENABLE_DIARIZATION = True in Section 2 to enable it.")
else:
    if not HUGGINGFACE_TOKEN:
        raise ValueError(
            "HUGGINGFACE_TOKEN is required for diarization. "
            "Set it in Section 2 and accept the pyannote model terms at: "
            "https://huggingface.co/pyannote/speaker-diarization-3.1"
        )

    import whisperx

    # ─────────────────────────────────────────────────────────────────────────
    # Step 1: Re-transcribe with WhisperX for alignment-compatible format
    # ─────────────────────────────────────────────────────────────────────────
    print("🔄  Loading WhisperX model for alignment...")
    wx_model = whisperx.load_model(
        MODEL_SIZE, DEVICE,
        compute_type=COMPUTE_TYPE,
        language=transcription_info.language
    )

    audio = whisperx.load_audio(AUDIO_FILE)
    wx_result = wx_model.transcribe(audio, batch_size=16)

    # ─────────────────────────────────────────────────────────────────────────
    # Step 2: Align word-level timestamps
    # ─────────────────────────────────────────────────────────────────────────
    print("🔗  Aligning word-level timestamps...")
    align_model, align_metadata = whisperx.load_align_model(
        language_code=wx_result["language"],
        device=DEVICE
    )
    wx_result = whisperx.align(
        wx_result["segments"],
        align_model,
        align_metadata,
        audio,
        DEVICE,
        return_char_alignments=False
    )

    # ─────────────────────────────────────────────────────────────────────────
    # Step 3: Run speaker diarization
    # ─────────────────────────────────────────────────────────────────────────
    print("👥  Running speaker diarization...")
    diarize_model = whisperx.DiarizationPipeline(
        use_auth_token=HUGGINGFACE_TOKEN,
        device=DEVICE
    )
    diarize_segments = diarize_model(
        audio,
        min_speakers=1,
        max_speakers=10
    )

    # ─────────────────────────────────────────────────────────────────────────
    # Step 4: Assign speakers to transcript segments
    # ─────────────────────────────────────────────────────────────────────────
    print("🏷️   Assigning speakers to segments...")
    wx_result = whisperx.assign_word_speakers(diarize_segments, wx_result)

    # Override segments with diarized version for export
    diarized_segments = wx_result["segments"]

    print("\n" + "─" * 60)
    print("DIARIZED TRANSCRIPTION:")
    print("─" * 60)

    for seg in diarized_segments:
        speaker = seg.get("speaker", "Unknown")
        start = seg["start"]
        end = seg["end"]
        text = seg["text"].strip()
        print(f"[{start:.2f}s → {end:.2f}s] [{speaker}]: {text}")

## 6️⃣ Preview de segmentos con timestamps

Vista previa de todos los segmentos antes de exportar.

In [ ]:
def format_timestamp(seconds: float) -> str:
    """Convert seconds to HH:MM:SS.mmm format."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = seconds % 60
    return f"{hours:02d}:{minutes:02d}:{secs:06.3f}"

print(f"{'#':<4} {'START':<14} {'END':<14} {'TEXT'}")
print("─" * 80)

for i, seg in enumerate(segments, 1):
    start_fmt = format_timestamp(seg.start)
    end_fmt   = format_timestamp(seg.end)
    text      = seg.text.strip()
    print(f"{i:<4} {start_fmt:<14} {end_fmt:<14} {text}")

## 7️⃣ Exportación de resultados

Genera los cuatro formatos de salida más comunes.

In [ ]:
import json
import srt
from datetime import timedelta
import os

# Base filename from the uploaded audio (without extension)
BASE_NAME = os.path.splitext(AUDIO_FILE)[0]

def seconds_to_timedelta(seconds: float) -> timedelta:
    """Convert float seconds to timedelta for SRT/VTT generation."""
    return timedelta(seconds=seconds)


def build_segments_dict(segs) -> list:
    """Convert faster-whisper segments to a serializable list of dicts."""
    result = []
    for seg in segs:
        entry = {
            "id": seg.id,
            "start": round(seg.start, 3),
            "end": round(seg.end, 3),
            "text": seg.text.strip(),
            "avg_logprob": round(seg.avg_logprob, 4),
            "no_speech_prob": round(seg.no_speech_prob, 4),
            "words": [
                {
                    "word": w.word,
                    "start": round(w.start, 3),
                    "end": round(w.end, 3),
                    "probability": round(w.probability, 4)
                }
                for w in (seg.words or [])
            ]
        }
        result.append(entry)
    return result


def export_txt(text: str, path: str):
    """Export plain transcription text."""
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"📄  TXT exported → {path}")


def export_json(segs, info, path: str):
    """Export full transcription with timestamps and metadata in JSON format."""
    output = {
        "metadata": {
            "language": info.language,
            "language_probability": round(info.language_probability, 4),
            "duration": round(info.duration, 3),
            "model": MODEL_SIZE
        },
        "text": full_text,
        "segments": build_segments_dict(segs)
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    print(f"🗂️   JSON exported → {path}")


def export_srt(segs, path: str):
    """Export subtitles in SRT format."""
    subtitles = [
        srt.Subtitle(
            index=i + 1,
            start=seconds_to_timedelta(seg.start),
            end=seconds_to_timedelta(seg.end),
            content=seg.text.strip()
        )
        for i, seg in enumerate(segs)
    ]
    with open(path, "w", encoding="utf-8") as f:
        f.write(srt.compose(subtitles))
    print(f"🎬  SRT exported → {path}")


def export_vtt(segs, path: str):
    """Export subtitles in WebVTT format (compatible with HTML5 video)."""
    def fmt_vtt(seconds: float) -> str:
        h = int(seconds // 3600)
        m = int((seconds % 3600) // 60)
        s = seconds % 60
        return f"{h:02d}:{m:02d}:{s:06.3f}".replace(".", ",").replace(",", ".")

    lines = ["WEBVTT", ""]
    for i, seg in enumerate(segs, 1):
        lines.append(str(i))
        lines.append(f"{fmt_vtt(seg.start)} --> {fmt_vtt(seg.end)}")
        lines.append(seg.text.strip())
        lines.append("")

    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"📺  VTT exported → {path}")


print("✅ Export helpers defined. Run the next cell to generate and download all files.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Generate all output files
# ─────────────────────────────────────────────────────────────────────────────
txt_path  = f"{BASE_NAME}_transcription.txt"
json_path = f"{BASE_NAME}_transcription.json"
srt_path  = f"{BASE_NAME}_subtitles.srt"
vtt_path  = f"{BASE_NAME}_subtitles.vtt"

export_txt(full_text, txt_path)
export_json(segments, transcription_info, json_path)
export_srt(segments, srt_path)
export_vtt(segments, vtt_path)

print("\n⬇️  Downloading all files...")
files.download(txt_path)
files.download(json_path)
files.download(srt_path)
files.download(vtt_path)

print("\n✅ Done! All files downloaded.")

## 8️⃣ Post-procesamiento con IA *(opcional)*

Usa la API de Groq (gratis) para limpiar, resumir o generar actas de reunión a partir de la transcripción.

In [ ]:
!pip install -q groq
print("✅ Groq SDK installed.")

In [ ]:
from groq import Groq

# ─────────────────────────────────────────────────────────────────────────────
# Configuration
# Get your free key at: https://console.groq.com
# ─────────────────────────────────────────────────────────────────────────────
GROQ_API_KEY = ""  # ← Insert your Groq API key here

# Choose the task:
#   "clean"    → Fix punctuation, capitalization and remove filler words
#   "summary"  → Generate a concise summary
#   "minutes"  → Generate meeting minutes with action items
POSTPROCESS_TASK = "clean"

# ─────────────────────────────────────────────────────────────────────────────
# System prompts per task
# ─────────────────────────────────────────────────────────────────────────────
SYSTEM_PROMPTS = {
    "clean": (
        "You are an expert transcript editor. Fix punctuation, capitalization, "
        "remove filler words (um, uh, like), and correct obvious transcription errors. "
        "Return ONLY the cleaned text, no explanations."
    ),
    "summary": (
        "You are an expert summarizer. Read the transcription and produce a clear, "
        "concise summary of the main points. Use bullet points. "
        "Respond in the same language as the transcription."
    ),
    "minutes": (
        "You are an expert meeting facilitator. From the transcription, generate structured "
        "meeting minutes with sections: Overview, Key Discussion Points, Decisions Made, "
        "and Action Items. Respond in the same language as the transcription."
    )
}

if not GROQ_API_KEY:
    print("⚠️  GROQ_API_KEY is empty. Get your free key at https://console.groq.com")
else:
    client = Groq(api_key=GROQ_API_KEY)

    print(f"🤖  Running post-processing task: '{POSTPROCESS_TASK}'...")

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPTS[POSTPROCESS_TASK]},
            {"role": "user",   "content": full_text}
        ],
        temperature=0.3,
        max_tokens=4096
    )

    postprocessed_text = response.choices[0].message.content

    # Save and download the post-processed output
    postprocess_path = f"{BASE_NAME}_{POSTPROCESS_TASK}.txt"
    with open(postprocess_path, "w", encoding="utf-8") as f:
        f.write(postprocessed_text)

    print(f"\n📋  Result ({POSTPROCESS_TASK}):")
    print("─" * 60)
    print(postprocessed_text)
    print("─" * 60)

    files.download(postprocess_path)
    print(f"\n✅  Downloaded: {postprocess_path}")

## 9️⃣ Transcripción directa con API de Groq *(alternativa sin GPU)*

Si no tienes GPU o el archivo es pequeño (< 25 MB), puedes usar Groq para transcribir
directamente con `whisper-large-v3-turbo` hospedado en su infraestructura.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Direct transcription via Groq API (no local GPU required)
# File size limit: 25 MB
# Speed: ~216x real-time
# ─────────────────────────────────────────────────────────────────────────────
GROQ_API_KEY_TRANSCRIBE = ""  # ← Insert your Groq API key here
GROQ_AUDIO_FILE = AUDIO_FILE  # Same uploaded file from Section 3

file_size_mb = os.path.getsize(GROQ_AUDIO_FILE) / (1024 * 1024)

if not GROQ_API_KEY_TRANSCRIBE:
    print("⚠️  Set GROQ_API_KEY_TRANSCRIBE to use this section.")
elif file_size_mb > 25:
    print(f"⚠️  File is {file_size_mb:.1f} MB — Groq API limit is 25 MB.")
    print("    Use the local faster-whisper transcription from Section 4 instead.")
else:
    from groq import Groq
    groq_client = Groq(api_key=GROQ_API_KEY_TRANSCRIBE)

    print(f"🚀  Transcribing via Groq API ({file_size_mb:.1f} MB)...")
    start = time.time()

    with open(GROQ_AUDIO_FILE, "rb") as audio_file:
        groq_result = groq_client.audio.transcriptions.create(
            file=(GROQ_AUDIO_FILE, audio_file.read()),
            model="whisper-large-v3-turbo",
            response_format="verbose_json",
            timestamp_granularities=["segment"],
            language=LANGUAGE  # None = auto-detect
        )

    elapsed_groq = time.time() - start
    print(f"✅  Done in {elapsed_groq:.1f}s")
    print(f"\n🌐  Language: {groq_result.language}")
    print("\n" + "─" * 60)
    print(groq_result.text)

    # Save Groq output
    groq_path = f"{BASE_NAME}_groq_transcription.json"
    with open(groq_path, "w", encoding="utf-8") as f:
        json.dump(groq_result.model_dump(), f, ensure_ascii=False, indent=2)
    files.download(groq_path)
    print(f"\n⬇️  Downloaded: {groq_path}")

---

## 📚 Referencia rápida

| Recurso | URL |
|---|---|
| faster-whisper | https://github.com/SYSTRAN/faster-whisper |
| WhisperX | https://github.com/m-bain/whisperX |
| Groq Console | https://console.groq.com |
| pyannote models | https://huggingface.co/pyannote |
| Whisper original | https://github.com/openai/whisper |

### Modelos disponibles en faster-whisper

| Modelo | VRAM | Velocidad relativa | WER |
|---|---|---|---|
| `tiny` | ~1 GB | Extrema | Alto |
| `base` | ~1 GB | Muy alta | Medio-alto |
| `small` | ~2 GB | Alta | Medio |
| `medium` | ~5 GB | Media | Bajo |
| `large-v3` | ~10 GB | Lenta | Muy bajo |
| `large-v3-turbo` | ~6 GB | Alta | Muy bajo |
| `distil-large-v3` | ~6 GB | Muy alta | Bajo |
